### Day 4 Assignment: Apache Spark Fundamentals

### Basic Tasks 

#### 1. Create a DataFrame

In [0]:
employees = [
    {"name": "John", "age": 30},
    {"name": "Alice", "age": 25},
    {"name": "Bob", "age": 35},
    {"name": "Charlie", "age": 40}
]
df = spark.createDataFrame(employees)

In [0]:
df.display()

#### 2. Read a CSV 


In [0]:
df_inferred = spark.read.csv("/Volumes/dev/demo/ex-volume/sales/sales.csv", header=True, inferSchema=True)
df_inferred.display()

In [0]:
from pyspark.sql.types import *

explicit_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("transaction_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("order_date", DateType(), True)
])
df_explicit = spark.read.csv("/Volumes/dev/demo/ex-volume/sales/sales.csv", header=True, schema=explicit_schema)
df_explicit.display()

In [0]:
# Compare schemas
df_inferred.printSchema()
df_explicit.printSchema()

In [0]:
print(f"\nAre schemas strictly identical? {df_inferred.schema == df_explicit.schema}")

The condition df_inferred.schema == df_explicit.schema evaluates to False due to small structural differences in how PySpark automatically infers schema vs. your explicit definition.

#### 3. Filter transformation + Action + Explanation

In [0]:
df_filtered = df_explicit.filter(df_explicit["total_amount"] > 1000)

In [0]:
df_filtered.dropna().display()

#### Why Nothing Ran Until the Action (Lazy Evaluation)

* **In Spark, Transformations are Lazy:** Operations like .filter(), .select(), or .withColumn() do not immediately read or process any data. Instead, Spark simply records these instructions into a Logical Execution Plan (DAG / Lineage) without loading records into memory or disk.

* **Only Actions Trigger Execution:** When an action (such as .count(), .show(), or .display()) is called, the Catalyst Optimizer evaluates the complete lineage, optimizes the execution plan, and then only executes the physical plan or computation. 

### Intermediate Tasks

#### 4. End-to-end ELT

In [0]:
input_path = "/Volumes/dev/demo/ex-volume/Book1.csv"
catalog = "dev"
schema = "demo"
table_name ="Book1"

In [0]:
#Extract
raw_df = spark.read.csv(input_path, header=True, inferSchema=True)
raw_df.display()

In [0]:
#Transform
from pyspark.sql.functions import current_timestamp
transformed_df = raw_df.filter(raw_df["amount"]>0).withColumn("ingestion_time",current_timestamp())

In [0]:
#Load
transformed_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

In [0]:
%sql
select * from dev.demo.Book1

#### 5. Read & Flatten Nested JSON

In [0]:
from pyspark.sql.functions import col, explode
# Read JSON into DataFrame
df_nested = spark.read.option("multiline", True).json("/Volumes/dev/demo/ex-volume/json_data_driver.json")

In [0]:
df_nested.display()

In [0]:

# Flattening
df_flattened = (
    df_nested
    .select(
        col("circuit"),
        col("driver.id").alias("driver_id"),        # Struct flattening (dot notation)
        col("driver.team").alias("driver_team"),    # Struct flattening (dot notation)
        explode(col("pit_stops")).alias("pit_stop") # Array flattening (explode)
    )
    .select(
        "circuit",
        "driver_id",
        "driver_team",
        col("pit_stop.lap").alias("pit_lap"),
        col("pit_stop.duration").alias("pit_duration_sec")
    )
)

df_flattened.display()

#### 6.Call .explain()

In [0]:
transformed_df.explain(mode="formatted")

### Advanced Tasks 

#### 7. Pandas to PySpark

In [0]:
import pandas as pd

df = pd.read_csv("/Volumes/dev/demo/ex-volume/sales/sales.csv")

df = df[df["total_amount"] > 1000]

df["final_amount"] = df["total_amount"] - df["discount_amount"]

# Calculate total spending by customer
summary = (
    df.groupby("customer_id")["final_amount"]
      .sum()
      .reset_index()
      .sort_values("final_amount", ascending=False)
)

print(summary)

In [0]:
from pyspark.sql import functions as F

df = spark.read.csv("/Volumes/dev/demo/ex-volume/sales/sales.csv", header=True, inferSchema=True)

df = df.filter(F.col("total_amount") > 1000)

df = df.withColumn(
    "final_amount",
    F.col("total_amount") - F.col("discount_amount")
)

# Calculate total spending by customer
summary = (
    df.groupBy("customer_id")
      .agg(
          F.round(F.sum("final_amount"),2).alias("final_amount")
      )
      .orderBy(F.col("final_amount").desc())
)

summary.show()

Pandas works well for smaller datasets because the data is generally loaded into the memory of a single machine. But, as the number of orders increases, this can become a limitation.

1. Memory limitation:
Pandas needs to keep the dataset in the memory of one machine. If the dataset becomes larger than the available RAM, the operation can become slow or fail. Spark distributes the data across multiple machines, so it can handle much larger datasets.

2. Single-machine processing:
Pandas performs the filtering and calculations on a single machine. Spark distributes these operations across worker nodes, allowing large datasets to be processed in parallel.

3. GroupBy and aggregation:
Operations such as groupby can become expensive when there are millions or billions of orders. Spark distributes the aggregation across the cluster instead of processing everything on one machine, making it more scalable.

#### 8. Partitioning/write strategy

In [0]:
from pyspark.sql import functions as F

df = spark.read.csv("/Volumes/dev/demo/ex-volume/sales/sales.csv", header=True, inferSchema=True)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .saveAsTable("dev.demo.sales_partitioned")

In [0]:
%sql
select * from dev.demo.sales_partitioned

In [0]:
%sql
SELECT *
FROM dev.demo.sales_partitioned
WHERE order_date BETWEEN '2023-12-01' AND '2023-12-30';
-- the query doesn't need to scan every partition. Spark can use the filter to eliminate partitions that don't match the requested dates. This is called partition pruning.

Lazy evaluation and physical plan

Spark transformations such as filter(), select(), and groupBy() are lazy. Spark does not execute them immediately. Instead, it builds a logical plan and later creates a physical plan when an action such as show(), count(), or write() is called.

In [0]:
df.filter(
    F.col("order_date").between("2026-08-20", "2026-08-22")
).explain(True) #inspecting physical plan

#### 9. 

In [0]:
from pyspark.sql import functions as F
df = spark.table("dev.demo.sales_partitioned")

In [0]:
monthly_revenue = (
    df.withColumn(
        "month",
        F.date_format(F.col("order_date"), "yyyy-MM")
    )
    .groupBy("month")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy("month")
)

In [0]:
monthly_revenue.display()

In [0]:
monthly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dev.demo.monthly_revenue_report")
    #This table can be imported for dashboard to view monthly revenue report